In [1]:
# import all libraries
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.utils.data import Dataset
import sys
import optuna
from functools import partial
from pathlib import Path
import gc
import warnings
import os
import matplotlib.pyplot as plt
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# change path to the project root
base_path = Path.cwd() / "../../"
sys.path.append(str(base_path.resolve()))

# load custom classes and functions
from utils.classification import EarlyStopping, train_bert, tune_bert_stance_optuna
from utils.evaluation import run_testset_stance

In [2]:
# load the data
with open("../../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

# initialize dictionary for the sentiment classes
sent_dict = set()

# loop through all sentences
for task in data:
    if task["annotations"]:
        for annotation in task["annotations"]:
            label = annotation["tag"][3:]
            sent_dict.add(label)

# sort the tag dictionary
label_list = sorted(sent_dict)

# dictionaries that convert from id to tag and vice versa
label_to_id = {tag: i for i, tag in enumerate(label_list)}
id_to_label = {id: label for label, id in label_to_id.items()}

In [3]:
# get all data with annotations
data_with_annotations = []
for task in data:
    if task["annotations"]:
        data_with_annotations.append(task)

# split into training and test dataset
train_data, val_data = train_test_split(data_with_annotations, test_size=0.2, shuffle=True, random_state=42)

class StanceDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len=128):
        self.dataset = []
        for item in data:
            sentence = item["sentence"]
            for ann in item["annotations"]:
                span_text = ann["text"]
                label = label2id[ann["tag"][3:]]
                # combine sentence and target span
                encoded = tokenizer(
                    sentence,
                    span_text,
                    truncation=True,
                    padding="max_length",
                    max_length=max_len,
                    return_tensors="pt"
                )
                self.dataset.append({
                    "input_ids": encoded["input_ids"].squeeze(0),
                    "attention_mask": encoded["attention_mask"].squeeze(0),
                    "label": torch.tensor(label, dtype=torch.long)
                })
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        return self.dataset[idx]

In [ ]:
# hyperparameter tuning with TPE implemented via Optuna

# define the objective function whose output optuna tries to maximize
def objective(trial, model_name, train_dataset, val_dataset, collate_fn, tag2id, id2tag, device, early_stopper):

    # hyperparameter space in which optuna can search
    params = {
        "lr": trial.suggest_categorical("lr", [9e-6, 2e-5, 4e-5]),
        "batch_size": trial.suggest_categorical("batch_size", [8, 16, 32]),
        "weight_decay": trial.suggest_categorical("weight_decay", [0.01, 0.1, 0.3]),
        "epochs": trial.suggest_categorical("epochs", [2, 2, 2])
    }

    # train the model with these hyperparameters
    best_f1, best_epoch, train_losses, val_losses, f1_scores_train, f1_scores_val = tune_bert_stance_optuna(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        collate_fn=collate_fn,
        model_name=model_name,
        tag2id=tag2id,
        id2tag=id2tag,
        device=device,
        early_stopper=early_stopper,
        params=params
    )

    # besides parameters, save losses and f1-scores in the trial object
    trial.set_user_attr("best_epoch", best_epoch)
    trial.set_user_attr("train_losses", train_losses)
    trial.set_user_attr("val_losses", val_losses)
    trial.set_user_attr("f1_scores_train", f1_scores_train)
    trial.set_user_attr("f1_scores_val", f1_scores_val)

    return best_f1


# dictionary to save optimal paramaters
optimal_configs = {}

# number of trials, the device on which models should operate and the specific models to test
num_trials = 2
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model_names = ["microsoft/deberta-v3-base", "roberta-base", "bert-base-cased", "distilbert-base-cased"]

# loop over the models
for model_name in model_names:

    print(f"\nStarting search for model: {model_name}")
    print("-"*200)

    # create a new optuna study, tokenizer fitting to the model as well as train and validation set
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset = StanceDataset(data=train_data, tokenizer=tokenizer, label2id=label_to_id, max_len=128)
    val_dataset = StanceDataset(data=val_data, tokenizer=tokenizer, label2id=label_to_id, max_len=128)

    # run the optimization loop
    study.optimize(
        partial(objective,
                model_name=model_name,
                train_dataset=train_dataset,
                val_dataset=val_dataset,
                label2id=label_to_id,
                id2label=id_to_label,
                device=device,
                early_stopper=EarlyStopping(patience=3, path=f"model_saves/{model_name}_checkpoint", printoption=True)),
        n_trials=num_trials
    )

    # get the best trial and save optimal parameters
    best_trial = study.best_trial
    optimal_configs[model_name] = {
        "best_f1": best_trial.value,
        "best_params": best_trial.params,
        "best_epoch": best_trial.user_attrs["best_epoch"],
        "train_losses": best_trial.user_attrs["train_losses"],
        "val_losses": best_trial.user_attrs["val_losses"],
        "f1_scores_train": best_trial.user_attrs["f1_scores_train"],
        "f1_scores_val": best_trial.user_attrs["f1_scores_val"]
    }

    # clean all objects and empty cache for the next model
    del study, tokenizer, train_dataset, val_dataset
    gc.collect()
    torch.mps.empty_cache()

    print("-"*200)

In [ ]:
# plot the development of F1 scores and losses for the tuned models

# as many rows as models and 2 columns for loss and f1
fig, axes = plt.subplots(len(model_names), 2, figsize=(12, 4 * len(model_names)))

# loop over the models
for i, model_name in enumerate(model_names):

    # get the data for this specific model
    config = optimal_configs[model_name]
    epochs = list(range(1, len(config['train_losses']) + 1))

    # create the loss subplot
    ax_loss = axes[i, 0]
    ax_loss.plot(epochs, config['train_losses'], marker='o', label='Train Loss')
    ax_loss.plot(epochs, config['val_losses'], marker='o', label='Validation Loss')
    ax_loss.set_title(f"{model_name} - Loss")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.legend()
    ax_loss.grid(True)

    # create the F1 subplot
    ax_f1 = axes[i, 1]
    ax_f1.plot(epochs, config['f1_scores_train'], marker='o', label='Train F1')
    ax_f1.plot(epochs, config['f1_scores_val'], marker='o', label='Validation F1')
    ax_f1.set_title(f"{model_name} - F1 Score")
    ax_f1.set_xlabel("Epoch")
    ax_f1.set_ylabel("F1 Score")
    ax_f1.legend()
    ax_f1.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# apply cross validation to all models with their best hyperparameters

# empty dictionary to store the final average metrics
average_metrics = {}

# loop over all individual models
for model_name in model_names:

    print(f"\nCross-validation for model {model_name} starts")
    print("-"*200)

    # dictionary to save the individual fold evaluation metrics
    fold_metrics = {
        "seqeval": [],
        "cross_span": [],
        "sentence_level": []
    }

    # get the optimal hyperparameters for this model
    epochs = optimal_configs[model_name]["best_epoch"]
    lr = optimal_configs[model_name]["best_params"]["lr"]
    batch_size = optimal_configs[model_name]["best_params"]["batch_size"]
    weight_decay = optimal_configs[model_name]["best_params"]["weight_decay"]

    # define the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # create cross-validation object for 5 folds
    k = 5
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    # randomly split the training data into 5 folds and loop over them
    for fold, (train_idx, val_idx) in enumerate(kf.split(train_data)):

        print(f"\nFold number: {fold + 1}")

        # create the training and validation fold based on the provided indices
        train_fold_data = [train_data[i] for i in train_idx]
        val_fold_data = [train_data[i] for i in val_idx]

        # create tensor dataset and respective data loaders
        train_dataset = TokenDataset(data=train_fold_data, tokenizer=tokenizer, tag2id=tag_to_id)
        val_dataset = TokenDataset(data=val_fold_data, tokenizer=tokenizer, tag2id=tag_to_id)
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
        val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

        # set the device explicitly
        device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

        # create the model and optimizer
        model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=len(tag_to_id),
            id2label=id_to_tag,
            label2id=tag_to_id
            ).to(device)
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

        # train the model with the optimal configuration
        train_bert(train_dataloader, model, optimizer, epochs, device, which_task="ner")

        # create dictionary to save the inputs for each evaluation metric
        evaluation_inputs = {
        "seqeval": {},
        "cross_span": {},
        "sentence_level": {}
        }
        
        # loop over all metrics and get the ground truth as well as predicted labels for the validation set
        for metric in evaluation_inputs.keys():
            all_true, all_pred, _ = run_testset_ner(
                model=model, test_dataloader=val_dataloader, id2tag=id_to_tag, device=device, for_metric=metric
                )
            evaluation_inputs[metric] = {
                "all_true": all_true,
                "all_pred":all_pred
                }
            
        # apply all evaluation functions and save in the dictionary
        metrics_seqeval = evaluate_seqeval(
        evaluation_inputs["seqeval"]["all_true"],
        evaluation_inputs["seqeval"]["all_pred"]
        )
        metrics_cross_span = mention_level_evaluation(
            evaluation_inputs["cross_span"]["all_true"],
            evaluation_inputs["cross_span"]["all_pred"]
        )
        metrics_sentence_level = sentence_level_evaluation(
            evaluation_inputs["sentence_level"]["all_true"],
            evaluation_inputs["sentence_level"]["all_pred"]
        )

        # append all metrics to the dictionary
        fold_metrics["seqeval"].append(metrics_seqeval)
        fold_metrics["cross_span"].append(metrics_cross_span)
        fold_metrics["sentence_level"].append(metrics_sentence_level)

    # calculate average metrics for all scores
    seqeval_scores = [m["f1"] for m in fold_metrics["seqeval"]]
    mean_seqeval = np.mean(seqeval_scores)
    std_seqeval = np.std(seqeval_scores)

    cross_span_scores = [m["f1"] for m in fold_metrics["cross_span"]]
    mean_cross_span = np.mean(cross_span_scores)
    std_cross_span = np.std(cross_span_scores)

    sentence_level_scores = [m["f1"] for m in fold_metrics["sentence_level"]]
    mean_sentence_level = np.mean(sentence_level_scores)
    std_sentence_level = np.std(sentence_level_scores)

    # append all metrics to the dictionary
    average_metrics[model_name] = {
        "seqeval": mean_seqeval,
        "cross_span": mean_cross_span,
        "sentence_level": mean_sentence_level
        }
    
    print("-"*200)

In [7]:
# models to train and empty dictionary to save results
model_names = ["roberta-base"]#, "bert-base-cased", "distilbert-base-cased"]
test_metrics = {}

# define global parameters
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
num_labels = len(label_to_id)

# loop
for model_name in model_names:

    # create new tokenizer depending on model
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # create new datasets and dataloaders for the specific model
    train_dataset_tensor = StanceDataset(train_dataset, tokenizer, label_to_id)
    test_dataset_tensor = StanceDataset(test_dataset, tokenizer, label_to_id)
    train_loader = DataLoader(train_dataset_tensor, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset_tensor, batch_size=16, shuffle=False)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)

    # define hyperparameters specific to the model 
    epochs = 20
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

    # train the model and directly run it through the test set
    train_bert(train_loader, model, optimizer, epochs, device, "sentiment")
    true_labels, pred_labels = run_testset_sentiment(model=model, test_dataloader=test_loader, device=device)

    # evaluate the results and save in dictionary
    metrics = classification_report(
        [list(label_to_id.keys())[i] for i in true_labels],
        [list(label_to_id.keys())[i] for i in pred_labels],
        output_dict=True
        )
    test_metrics[model_name] = {
        "negative": metrics["neg"]["f1-score"],
        "neutral": metrics["neutral"]["f1-score"],
        "positive": metrics["pos"]["f1-score"]
    }

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/20


Training: 100%|██████████| 103/103 [00:44<00:00,  2.33it/s, loss=0.489]


Average training loss: 0.7292
Epoch 2/20


Training: 100%|██████████| 103/103 [00:43<00:00,  2.35it/s, loss=0.25] 


...

Average training loss: 0.0074
Epoch 19/20


Training: 100%|██████████| 103/103 [00:43<00:00,  2.35it/s, loss=0.000104]


Average training loss: 0.0189
Epoch 20/20


Training: 100%|██████████| 103/103 [00:43<00:00,  2.36it/s, loss=7.33e-5]


Average training loss: 0.0265


In [8]:
# export the test metrics
test_metrics

{'roberta-base': {'negative': 0.7096774193548387,
  'neutral': 0.5867768595041323,
  'positive': 0.8114901256732495}}